# 12. Batch Processing & Workflow — 대량 처리

수백 개 HWP 문서를 처리할 때 필요한 **context manager** 를 활용한 고성능
워크플로우.

## 핵심 context manager 6종

| Context | 용도 | 효과 |
|:---|:---|:---|
| `silenced("yes")` | 다이얼로그 자동 응답 | "저장?" 등 YES 자동 |
| `suppress_errors()` | 에러 + Python 예외 swallow | 한두 파일 실패해도 계속 |
| `batch_mode(hide=True)` | 창 숨김 + dialog 억제 | **5~10배 가속** |
| `undo_group("설명")` | 단일 undo 경계 | 사용자가 Ctrl+Z 한 번으로 rollback |
| `charshape_scope(**fmt)` | 문자 모양 임시 변경 | 종료 시 snapshot 복원 |
| `use_document(doc)` | 활성 문서 일시 전환 | 블록 내 doc 기준으로 동작 |


## 1. silenced — 다이얼로그 자동 응답

```python
from hwpapi.core import App

app = App(is_visible=True)

# 기본: 모든 dialog 의 첫 버튼 자동 클릭
with app.silenced():                 # = silenced("yes")
    for path in paths:
        app.open(path)
        app.replace_all("2025", "2026")
        app.save(path)               # "덮어쓸까요?" → YES 자동
# 종료 시 이전 mode 자동 복원

# NO 로 응답
with app.silenced("no"):
    for doc in app.documents:
        doc.close()                  # "저장?" → NO → 저장 안 함

# 세밀 제어 — 저장만 자동
with app.silenced("save_yes"):
    app.save(path)

# 직접 비트필드
with app.silenced(0x111111):
    ...
```

사용 가능한 string preset:
`"yes"`, `"no"`, `"reset"`, `"ok"`, `"save"`, `"save_yes"`, `"save_no"`,
`"no_save"`, `"okcancel_ok"`, `"okcancel_no"`


## 2. suppress_errors — 에러도 무시하고 루프 계속

```python
from pathlib import Path
from hwpapi.core import App

app = App(is_visible=False)

failed = []
with app.silenced():
    for path in Path("input/").glob("*.hwp"):
        try:
            with app.suppress_errors():    # 내부 예외를 warning 으로 전환
                app.open(path)
                app.replace_all("old", "new")
                app.save(str(path).replace("input", "output"))
        except Exception as e:
            # suppress_errors 가 Python 예외를 삼키지만 외부 try/except
            # 로도 한 번 더 감싸면 안전
            failed.append((path, str(e)))

print(f"실패: {len(failed)}")
for p, e in failed:
    print(f"  {p}: {e}")
```


## 3. batch_mode — 5~10배 가속

수백 개 문서 일괄 편집 시 필수.

```python
import pandas as pd
from pathlib import Path
from hwpapi.core import App

app = App(is_visible=True)   # 호출 시 자동 숨김
Path("out").mkdir(exist_ok=True)

customers = pd.read_csv("customers.csv")

with app.batch_mode():                          # 창 숨김 + dialog off + scroll off
    for _, row in customers.iterrows():
        app.open("contract_template.hwp")
        app.fields.from_brackets()              # {{name}} → 필드 변환
        app.fields.update(row.to_dict())        # 일괄 값 주입
        app.save(f"out/contract_{row['name']}.hwp")
# 블록 종료 — 창 복원, mode 복원
```

**내부 동작**:
- `set_message_box_mode(SILENCE_ALL_YES)` — dialog 억제
- `set_visible(False)` — 창 숨김 (hide=True 일 때)
- `Run("FollowActiveWindowOff")` — scroll follow 중지

hide=False 면 dialog 만 억제:
```python
with app.batch_mode(hide=False):
    # 화면 보며 진행 (디버깅 중)
    ...
```


## 4. undo_group — 단일 Ctrl+Z 로 전체 rollback

```python
# 사용자가 대량 수정 후 "어.. 되돌리고 싶다" 할 때 유용
with app.undo_group("전체 폰트 교체"):
    app.convert.replace_font("맑은 고딕", "함초롬바탕")
    app.set_charshape(fontsize=1100)
    for para_idx in range(100):
        app.move.para.next()
        app.set_parashape(line_spacing=160)

# 사용자가 Ctrl+Z 한 번 누르면 위 모든 작업이 원복
```


## 5. charshape_scope / parashape_scope — 임시 서식

스냅샷 + 복원 패턴. shade_color 같은 "지우기 어려운" 속성도 안전.

```python
app.insert_text("앞: ")

# 이 블록 내부만 굵은 빨간 글씨
with app.charshape_scope(bold=True, text_color="#FF0000"):
    app.insert_text("중요")

app.insert_text(" 뒤")
# 결과: "앞: 중요 뒤" — "중요" 만 강조, 앞뒤는 기본 서식 유지

# 문단 모양도 동일
with app.parashape_scope(align_type=1, indent=1000):   # 중앙 정렬 + 들여쓰기
    app.insert_text("특별 문단")
```

**또는 더 간결한 Fluent 변형** — `styled_text()`:

```python
app.insert_text("앞: ")
app.styled_text("중요", bold=True, text_color="#FF0000")
app.insert_text(" 뒤")
# 동일한 결과 — 내부적으로 snapshot+restore 사용
```


## 6. use_document — 다중 문서 컨텍스트 전환

```python
doc1 = app.documents.open("report1.hwp")
doc2 = app.documents.open("report2.hwp")

# doc1 이 현재 활성
app.insert_text("doc1 에 삽입")

with app.use_document(doc2):
    # 블록 내에서는 doc2 가 활성
    app.insert_text("doc2 에 삽입")
    app.fields.update({"name": "홍길동"})

# 블록 종료 — doc1 다시 활성
app.insert_text("doc1 에 다시 삽입")
```

또는 xlwings 스타일 proxy 로:

```python
# doc.method() 는 자동으로 해당 doc 을 활성화한 뒤 app.method() 호출
doc1.insert_text("doc1")
doc2.insert_text("doc2")
# 사용자가 use_document() 직접 쓸 필요 없음
```


## 고급 조합 — 실전 workflow

### 실전 1 — 급여명세서 1000장 생성

```python
import pandas as pd
from pathlib import Path
from hwpapi.core import App

def generate_payslips(csv_path: str, template: str, out_dir: str):
    df = pd.read_csv(csv_path)   # name, id, gross, tax, net 등
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    failed = []
    app = App(is_visible=False)

    # 3중 context: batch_mode + silenced + suppress_errors
    with app.batch_mode():
        for _, row in df.iterrows():
            with app.suppress_errors():
                app.open(template)
                app.fields.from_brackets()
                app.fields.update(row.to_dict())
                out = Path(out_dir) / f"payslip_{row['id']}.hwp"
                app.save(str(out))
                app.close()

    print(f"완료: {len(df)} 건")
```

### 실전 2 — 전체 디렉터리의 폰트 교체

```python
from pathlib import Path
from hwpapi.core import App

def replace_font_recursive(root_dir: str, old: str, new: str):
    app = App(is_visible=False)
    with app.batch_mode():
        for path in Path(root_dir).rglob("*.hwp"):
            try:
                app.open(str(path))
                with app.undo_group(f"Font replace {old} → {new}"):
                    app.convert.replace_font(old, new)
                app.save()
                app.close()
                print(f"✓ {path.relative_to(root_dir)}")
            except Exception as e:
                print(f"✗ {path.relative_to(root_dir)}: {e}")
```

### 실전 3 — 문서 품질 일괄 리포트

```python
from pathlib import Path
from hwpapi.core import App

def lint_directory(root_dir: str) -> list:
    app = App(is_visible=False)
    results = []
    with app.batch_mode():
        for path in Path(root_dir).rglob("*.hwp"):
            app.open(str(path))
            r = app.lint()
            if r.has_issues:
                results.append({
                    "path": str(path),
                    "issues": r.issue_count,
                    "long_sentences": len(r.long_sentences),
                    "empty_paragraphs": len(r.empty_paragraphs),
                })
            app.close()
    return sorted(results, key=lambda x: -x["issues"])
```
